# Crop Classification with ResNet50

This notebook trains a ResNet50 model for crop classification and saves it in multiple formats.

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import pickle
import os

print(f"TensorFlow version: {tf.__version__}")

## Setup and Configuration

In [ ]:
# Configuration
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 50
NUM_CLASSES = 10  # Update based on your dataset

# Paths - Update these paths to your dataset location
TRAIN_DIR = 'path/to/train/data'
VAL_DIR = 'path/to/validation/data'

# Model save paths
MODEL_SAVE_DIR = os.path.dirname(os.path.abspath('__file__')) if '__file__' in dir() else './'
KERAS_MODEL_PATH = os.path.join(MODEL_SAVE_DIR, 'Crop_CLF_V1.keras')
PKL_MODEL_PATH = os.path.join(MODEL_SAVE_DIR, 'crop_model.pkl')
CLASS_NAMES_PATH = os.path.join(MODEL_SAVE_DIR, 'class_names.pkl')

## Data Preparation

In [ ]:
# Data augmentation for training
train_datagen = ImageDataGenerator(
    preprocessing_function=tf.keras.applications.resnet50.preprocess_input,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

# Only preprocessing for validation
val_datagen = ImageDataGenerator(
    preprocessing_function=tf.keras.applications.resnet50.preprocess_input
)

# Load training data
train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

# Load validation data
validation_generator = val_datagen.flow_from_directory(
    VAL_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

# Get class names
class_names = {v: k for k, v in train_generator.class_indices.items()}
NUM_CLASSES = len(class_names)
print(f"Number of classes: {NUM_CLASSES}")
print(f"Class names: {class_names}")

## Model Architecture

In [ ]:
# Load pre-trained ResNet50 model
base_model = ResNet50(
    weights='imagenet',
    include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

# Freeze base model layers
base_model.trainable = False

# Build the model
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(512, activation='relu')(x)
x = Dropout(0.5)(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.3)(x)
predictions = Dense(NUM_CLASSES, activation='softmax')(x)

# Create final model
model = Model(inputs=base_model.input, outputs=predictions)

# Compile the model
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

## Training Callbacks

In [ ]:
# Define callbacks
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.2,
    patience=3,
    min_lr=0.00001,
    verbose=1
)

callbacks = [early_stopping, reduce_lr]

## Model Training

In [ ]:
# Train the model
history = model.fit(
    train_generator,
    steps_per_epoch=train_generator.samples // BATCH_SIZE,
    validation_data=validation_generator,
    validation_steps=validation_generator.samples // BATCH_SIZE,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1
)

## Fine-tuning (Optional)

Unfreeze some layers of the base model for fine-tuning

In [ ]:
# Fine-tuning: Unfreeze the last few layers of base_model
base_model.trainable = True

# Freeze all layers except the last 20
for layer in base_model.layers[:-20]:
    layer.trainable = False

# Recompile with lower learning rate
model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Continue training
history_fine = model.fit(
    train_generator,
    steps_per_epoch=train_generator.samples // BATCH_SIZE,
    validation_data=validation_generator,
    validation_steps=validation_generator.samples // BATCH_SIZE,
    epochs=20,
    callbacks=callbacks,
    verbose=1
)

## Save the Model

Save the trained model in multiple formats for production use

In [ ]:
print("\n" + "="*60)
print("SAVING MODEL IN MULTIPLE FORMATS")
print("="*60 + "\n")

# 1. Save as Keras model (.keras format - recommended for TensorFlow 2.x)
print(f"Saving Keras model to: {KERAS_MODEL_PATH}")
model.save(KERAS_MODEL_PATH)
print("✓ Keras model saved successfully!\n")

# 2. Save as pickle file (for compatibility)
print(f"Saving pickle model to: {PKL_MODEL_PATH}")
model_data = {
    'model': model,
    'class_names': class_names,
    'img_size': IMG_SIZE,
    'num_classes': NUM_CLASSES
}
with open(PKL_MODEL_PATH, 'wb') as f:
    pickle.dump(model_data, f)
print("✓ Pickle model saved successfully!\n")

# 3. Save class names dictionary separately
print(f"Saving class names to: {CLASS_NAMES_PATH}")
with open(CLASS_NAMES_PATH, 'wb') as f:
    pickle.dump(class_names, f)
print("✓ Class names saved successfully!\n")

print("="*60)
print("ALL MODELS SAVED SUCCESSFULLY!")
print("="*60)
print(f"\nFiles created:")
print(f"1. {KERAS_MODEL_PATH}")
print(f"2. {PKL_MODEL_PATH}")
print(f"3. {CLASS_NAMES_PATH}")
print(f"\nClass names: {class_names}")

## Model Evaluation

In [ ]:
# Evaluate the model
test_loss, test_accuracy = model.evaluate(validation_generator)
print(f"\nTest Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")

## Test Prediction

Test the saved model by loading and making a prediction

In [ ]:
# Test loading the saved model
print("\nTesting model loading...\n")

# Load the keras model
loaded_model = tf.keras.models.load_model(KERAS_MODEL_PATH)
print("✓ Keras model loaded successfully!")

# Load class names
with open(CLASS_NAMES_PATH, 'rb') as f:
    loaded_class_names = pickle.load(f)
print(f"✓ Class names loaded: {loaded_class_names}")

print("\nModel ready for production use!")